# Reproduction of ICD for HIK SVM
**Attempting to reproduce:** The core "Intersection Coordinate Descent" (ICD) Algorithm 2 from the paper for L2-loss Dual HIK SVM training.
**Evaluation Metric:** Binary Classification Accuracy.

This block initializes the dual coordinate descent boundaries and constants.
*   **Cites:** Section 3.3, Equation 11 variables where $Q_{ii} = y_i y_{i'} B(x_i)^T B(x_{i'}) + D_{ii}$. In self-intersection $x_i = x_{i'}$, we get sum over dimensions of $x_i$, plus regularizer $D_{ii} = 1 / (2C)$.

This block physically wraps the main algorithm iteration cycle and evaluation loop.
*   **Cites:** Algorithm 2 (ICD method bounds mapping mapped out mathematically to vectorized exact matrix numpy operations). Evaluating predictions follows Section 3.2, Equation 6 exactly referencing the dynamic table mapping `T`.

In [1]:
import numpy as np

def train_icd(X, y, C=0.001, v_bar=100, max_iter=30):
    n, d = X.shape
    # line 1': Initialize Table T to zero
    T = np.zeros((d, v_bar + 1))
    alpha = np.zeros(n)
    D_ii = 1.0 / (2 * C)
    
    # Q_bar_ii for self-intersection calculation
    # min(xi_j, xi_j) is just xi_j
    Q_bar_ii = np.sum(X, axis=1) + D_ii
    
    # Dual coordinate descent loop over iterations
    for it in range(max_iter):
        for i in range(n):
            # line 4': G computation
            # \sum_{j} T_{j, xi_j} equivalent
            G_sum = np.sum(T[np.arange(d), X[i, :]])
            G = y[i] * G_sum - 1 + D_ii * alpha[i]
            
            # Projected Gradient
            PG = min(G, 0) if alpha[i] == 0 else G
            
            if abs(PG) > 1e-12:
                alpha_old = alpha[i]
                alpha[i] = max(alpha[i] - G / Q_bar_ii[i], 0)
                
                # delta scale
                delta = (alpha[i] - alpha_old) * y[i]
                
                # line 9': exact Table T update
                for j in range(d):
                    # T_{j,k} += delta * min(xi_j, k)
                    k_vals = np.arange(v_bar + 1)
                    T[j, :] += delta * np.minimum(X[i, j], k_vals)
                    
    return T, alpha

def predict_icd(X_test, T):
    n_test = X_test.shape[0]
    preds = np.zeros(n_test)
    for i in range(n_test):
        # Equation 6 equivalent for scoring
        preds[i] = np.sum(T[np.arange(X_test.shape[1]), X_test[i, :]])
    return np.sign(preds)
